# FYS4480/FYS9480 -- Week 42: the electron gas, and the density as the variable

Companion notebook to the week-42 slides (lecture notes chapter 6, section
6.14, and chapter 8, sections 8.1-8.4).  Plain NumPy throughout.

1. The Hartree-Fock single-particle energy of the electron gas: the closed
   form checked against numerical quadrature, the function $F(x)$, the band
   width and the effective mass that goes to zero logarithmically.
2. The energy per electron $2.21/r_s^2 - 0.916/r_s$ Ry, its coefficients,
   its minimum, and the correlation energy that it misses.
3. Thomas-Fermi from the electron gas: the local kinetic functional against
   the exact kinetic energy of $N$ spinless fermions in a harmonic trap, and
   the shell structure that no functional of the density alone can see.

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. The single-particle energy

With $x = k/k_F$ and
$F(x)=\tfrac12+\tfrac{1-x^2}{4x}\ln\left|\tfrac{1+x}{1-x}\right|$,
$\varepsilon^{\rm HF}_k/\varepsilon_F^0 = x^2 - 0.663\,r_s\,F(x)$, where
$0.663 = 4/(\pi k_F a_0)\cdot r_s^{-1} = 4/(\pi\,(9\pi/4)^{1/3})$.

In [2]:
def F(x):
    x = np.asarray(x, dtype=float)
    out = np.full_like(x, 0.5)
    m = (x != 1.0) & (x != 0.0)
    out[m] = 0.5 + (1 - x[m]**2) / (4 * x[m]) * np.log(np.abs((1 + x[m]) / (1 - x[m])))
    out[x == 0.0] = 1.0
    return out

kF_r0 = (9 * np.pi / 4)**(1 / 3)          # k_F r_0 = 1.9192
coef = 4 / (np.pi * kF_r0)                 # 0.663
print("k_F r_0 = %.4f,   4/(pi k_F a_0) = %.4f r_s   (the lecture notes round this to 0.663)" % (kF_r0, coef))

def eps_hf(x, rs):
    return x**2 - coef * rs * F(x)

rs = 4.0
print("\n r_s = 4:  x      F(x)       eps_HF/eps_F   x^2")
for x in (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0):
    print("        %4.2f   %.6f   %+.6f   %.4f" % (x, F(x), eps_hf(x, rs), x**2))

k_F r_0 = 1.9192,   4/(pi k_F a_0) = 0.6634 r_s   (the lecture notes round this to 0.663)

 r_s = 4:  x      F(x)       eps_HF/eps_F   x^2
        0.00   1.000000   -2.653746   0.0000
        0.25   0.978899   -2.535249   0.0625
        0.50   0.911980   -2.170162   0.2500
        0.75   0.783779   -1.517449   0.5625
        1.00   0.500000   -0.326873   1.0000
        1.25   0.252812   +0.891601   1.5625
        1.50   0.164700   +1.812927   2.2500
        2.00   0.088020   +3.766416   4.0000


### The closed form against numerical quadrature

Step 4 of the derivation on the slides: the exchange self-energy is
$\frac{e^2}{k\pi}\int_0^{k_F} p\,\ln\left|\frac{p+k}{p-k}\right| dp$.  We
integrate it numerically (the integrand has an integrable logarithmic
singularity at $p=k$, which the midpoint rule handles) and compare with the
bracket $kk_F + \frac{k_F^2-k^2}{2}\ln\left|\frac{k+k_F}{k-k_F}\right|$.

In [3]:
def exchange_closed(k, kF=1.0):
    return k * kF + 0.5 * (kF**2 - k**2) * np.log(abs((k + kF) / (k - kF)))

def exchange_quadrature(k, kF=1.0, n=400001):
    p = (np.arange(n) + 0.5) * kF / n          # midpoint rule avoids p = k exactly
    return np.sum(p * np.log(np.abs((p + k) / (p - k)))) * kF / n

for k in (0.3, 0.9, 1.7):
    print("k = %.1f k_F:  closed form %.8f   quadrature %.8f" % (k, exchange_closed(k), exchange_quadrature(k)))

k = 0.3 k_F:  closed form 0.58166284   quadrature 0.58166272
k = 0.9 k_F:  closed form 1.17972170   quadrature 1.17972026
k = 1.7 k_F:  closed form 0.42431925   quadrature 0.42431925


### Band width, and the effective mass

$F(0)=1$, $F(1)=\tfrac12$, so the occupied band has width
$1 + 0.3315\,r_s$ in units of $\varepsilon_F^0$.  The effective mass
$m^*/m = 2/(2 - 0.663\,r_s F'(1))$ needs $F'$ at the Fermi surface, which
diverges logarithmically; we evaluate it by a centred difference at
$1 - x = 10^{-2},\dots,10^{-8}$ and watch $m^*$ creep towards zero.

In [4]:
print("band width at r_s = 4: %.3f = %.3f - (%.3f)" % (eps_hf(1.0, rs) - eps_hf(0.0, rs), eps_hf(1.0, rs), eps_hf(0.0, rs)))
print("general: 1 + %.4f r_s" % (coef / 2))

def dF(x, h=1e-9):
    return (F(x + h) - F(x - h)) / (2 * h)

print("\n   1-x      F'(x)        m*/m      m* ln(1/(1-x))")
for d in (1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8):
    x = 1 - d
    mstar = 2 / (2 - coef * rs * dF(x, h=min(1e-9, d / 100)))
    print("  %.0e   %9.4f   %.4f     %.3f" % (d, dF(x, h=min(1e-9, d / 100)), mstar, mstar * np.log(1 / d)))
print("\nThe product m* ln(1/(1-x)) stays of order one: m* ~ 1/ln(1-x), no plateau, no limit.")

band width at r_s = 4: 2.327 = -0.327 - (-2.654)
general: 1 + 0.3317 r_s

   1-x      F'(x)        m*/m      m* ln(1/(1-x))
  1e-01     -1.0893   0.4089     0.942
  1e-02     -2.1685   0.2579     1.188
  1e-03     -3.3035   0.1858     1.283
  1e-04     -4.4522   0.1448     1.333
  1e-05     -5.6031   0.1186     1.365
  1e-06     -6.7543   0.1004     1.387
  1e-07     -7.9056   0.0870     1.403
  1e-08     -9.0569   0.0768     1.415

The product m* ln(1/(1-x)) stays of order one: m* ~ 1/ln(1-x), no plateau, no limit.


## 2. The energy per electron

$E_0/N = \tfrac{3}{5}\varepsilon_F - \tfrac{3e^2k_F}{4\pi}$, which in
Rydberg ($e^2/2a_0$) and with $k_F a_0 = 1.9192/r_s$ reads
$2.21/r_s^2 - 0.916/r_s$.

In [5]:
c_kin = 3 / 5 * kF_r0**2       # 3/5 (9 pi/4)^{2/3}
c_x = 3 / (2 * np.pi) * kF_r0  # 3/(2 pi) (9 pi/4)^{1/3}
print("kinetic coefficient %.4f, exchange coefficient %.4f" % (c_kin, c_x))

def E_per_N(rs):
    return c_kin / rs**2 - c_x / rs

print("\n  r_s    E_0/N [Ry]")
for r in (1.0, 2.0, 3.0, 4.0, 4.83, 6.0, 8.0):
    print("  %4.2f   %+.6f" % (r, E_per_N(r)))
rs_min = 2 * c_kin / c_x
print("\nminimum at r_s = 2 x %.4f / %.4f = %.3f,  E_0/N = %.4f Ry = %.2f eV"
      % (c_kin, c_x, rs_min, E_per_N(rs_min), E_per_N(rs_min) * 13.606))
print("zero crossing at r_s = %.3f" % (c_kin / c_x))
print("flatness: E(4) = %.4f, E(4.83) = %.4f, E(6) = %.4f  ->  within %.1f%% over 4 <= r_s <= 6"
      % (E_per_N(4), E_per_N(rs_min), E_per_N(6), 100 * (1 - E_per_N(4) / E_per_N(rs_min))))

kinetic coefficient 2.2099, exchange coefficient 0.9163

  r_s    E_0/N [Ry]
  1.00   +1.293571
  2.00   +0.094310
  3.00   -0.059899
  4.00   -0.090964
  4.83   -0.094988
  6.00   -0.091336
  8.00   -0.080012

minimum at r_s = 2 x 2.2099 / 0.9163 = 4.823,  E_0/N = -0.0950 Ry = -1.29 eV
zero crossing at r_s = 2.412
flatness: E(4) = -0.0910, E(4.83) = -0.0950, E(6) = -0.0913  ->  within 4.2% over 4 <= r_s <= 6


### What is missing: the correlation energy

The diffusion Monte Carlo results of Ceperley and Alder, as parametrised by
Perdew and Zunger (1981) for the unpolarised gas, give the correlation
energy per electron in Rydberg.  It is about $-0.06$ Ry near the
Hartree-Fock minimum -- comparable to the whole Hartree-Fock binding -- and
it shifts the minimum of $E_0/N$ noticeably although it is almost flat in
$r_s$.  (This parametrisation is the one the local density approximation of
week 43 uses.)

In [6]:
def eps_c_PZ(rs):
    """Perdew-Zunger fit to Ceperley-Alder, unpolarised, in Rydberg (x2 from Hartree)."""
    rs = np.asarray(rs, dtype=float)
    out = np.empty_like(rs)
    hi = rs >= 1
    out[hi] = -0.1423 / (1 + 1.0529 * np.sqrt(rs[hi]) + 0.3334 * rs[hi])
    lo = ~hi
    out[lo] = -0.0480 + 0.0311 * np.log(rs[lo]) - 0.0116 * rs[lo] + 0.0020 * rs[lo] * np.log(rs[lo])
    return 2 * out

grid = np.linspace(1.5, 10, 8501)
E_hf = E_per_N(grid)
E_tot = E_hf + eps_c_PZ(grid)
print("  r_s    E_HF/N     eps_c      sum   [Ry]")
for r in (2.0, 3.0, 4.0, 4.83, 6.0):
    print("  %4.2f   %+.4f   %+.4f   %+.4f" % (r, E_per_N(r), eps_c_PZ(np.array([r]))[0], E_per_N(r) + eps_c_PZ(np.array([r]))[0]))
print("\nHartree-Fock minimum: r_s = %.2f, E = %.4f Ry" % (grid[E_hf.argmin()], E_hf.min()))
print("with correlation:     r_s = %.2f, E = %.4f Ry" % (grid[E_tot.argmin()], E_tot.min()))
print("the binding roughly doubles and the equilibrium density moves, while the mechanism is unchanged")

  r_s    E_HF/N     eps_c      sum   [Ry]
  2.00   +0.0943   -0.0902   +0.0041
  3.00   -0.0599   -0.0744   -0.1343
  4.00   -0.0910   -0.0641   -0.1551
  4.83   -0.0950   -0.0578   -0.1528
  6.00   -0.0913   -0.0510   -0.1423

Hartree-Fock minimum: r_s = 4.82, E = -0.0950 Ry
with correlation:     r_s = 4.18, E = -0.1553 Ry
the binding roughly doubles and the equilibrium density moves, while the mechanism is unchanged


## 3. Thomas-Fermi from the electron gas, in a one-dimensional trap

For spinless fermions in one dimension the local Fermi momentum is
$k_F(x)=\pi n(x)$ and the kinetic energy per length of the uniform gas is
$\hbar^2 k_F^3/(6\pi m)$, so the Thomas-Fermi functional is
$T_{\rm TF}[n]=\frac{\pi^2\hbar^2}{6m}\int n(x)^3\,dx$.  We evaluate it on
the exact density of $N$ particles filling the lowest oscillator orbitals
($\hbar=m=\omega=1$) and compare with the exact kinetic energy, which by the
virial theorem is half the total, $T=\frac12\sum_{k<N}(k+\tfrac12)=N^2/4$.

In [7]:
from numpy.polynomial.hermite import hermval
from math import factorial

x = np.linspace(-8, 8, 4001)
dx = x[1] - x[0]

def orbital(k, x):
    c = np.zeros(k + 1); c[k] = 1.0
    return hermval(x, c) * np.exp(-x**2 / 2) / np.sqrt(2.0**k * factorial(k) * np.sqrt(np.pi))

def density(N, x):
    return sum(orbital(k, x)**2 for k in range(N))

print("  N    T exact    T Thomas-Fermi   ratio    difference")
for N in (1, 2, 3, 4, 6, 8):
    n = density(N, x)
    T_tf = np.pi**2 / 6 * np.sum(n**3) * dx
    T_ex = N**2 / 4
    print("  %d   %9.6f   %12.6f   %.4f    %.3f" % (N, T_ex, T_tf, T_tf / T_ex, T_tf - T_ex))
print("\nThe ratio improves with N; the absolute error creeps up.  A fixed finite system never gets it back.")

  N    T exact    T Thomas-Fermi   ratio    difference
  1    0.250000       0.302300   1.2092    0.052
  2    1.000000       1.074844   1.0748    0.075
  3    2.250000       2.340025   1.0400    0.090
  4    4.000000       4.101805   1.0255    0.102
  6    9.000000       9.120038   1.0133    0.120
  8   16.000000      16.134286   1.0084    0.134

The ratio improves with N; the absolute error creeps up.  A fixed finite system never gets it back.


### Shell structure

The exact density of eight particles has eight maxima, the nodal structure
of the orbitals surviving the sum; Thomas-Fermi, having no orbitals, cannot
produce them.  We locate the maxima and the classical turning point of the
highest occupied level, $x=\sqrt{2\varepsilon_7}=\sqrt{15}$.

In [8]:
n8 = density(8, x)
interior = (n8[1:-1] > n8[:-2]) & (n8[1:-1] > n8[2:])
maxima = x[1:-1][interior]
print("maxima of n(x) for N = 8:", np.round(maxima, 2))
print("n at the innermost maxima: %.3f,  at the central minimum: %.3f" % (n8[np.argmin(np.abs(x - maxima[3]))], n8[np.argmin(np.abs(x))]))
print("classical turning point sqrt(15) = %.2f;  n(3.87) = %.4f, n(5) = %.1e" % (np.sqrt(15), n8[np.argmin(np.abs(x - 3.87))], n8[np.argmin(np.abs(x - 5))]))

maxima of n(x) for N = 8: [-2.93 -1.98 -1.16 -0.38  0.38  1.16  1.98  2.93]
n at the innermost maxima: 1.308,  at the central minimum: 1.234
classical turning point sqrt(15) = 3.87;  n(3.87) = 0.2046, n(5) = 5.7e-04
